In [2]:
import pandas as pd
import numpy as np
import logging
from os import listdir

import sys
sys.path.insert(1, '..')

#from utils.utils import get_all_countries, get_empty_country_df, test_df
#from utils.countryconverter import convert_country_df

## MAIN ODA

In [232]:
ODA_OECD_PATH = "../../data/raw/oda/OECD.DCD.FSD,DSD_DAC2@DF_DAC2A,1.3+all.csv"
ODA_CHINA_PATH = "../../data/raw/oda/china/AidDatas_Global_Chinese_Development_Finance_Dataset_Version_3_0/AidDatas_Global_Chinese_Development_Finance_Dataset_Version_3_0/AidDatasGlobalChineseDevelopmentFinanceDataset_v3.0.xlsx"
ODA_RUSSIACHINA_PATH = "../../data/raw/oda/oda_ru_china.xlsx"
ODA_INDIA_PATH = "../../data/raw/oda/india/ind_aid_global_country_releaseV1.xlsx"
CIA_RENAME_DICT = {'YEAR':'year', 'DONOR':'ego', 'Recipient':'alter', 'Value (millions of dollars)':'value'}
AIDDATA_RENAME_DICT = {'Financier Country':'ego', 'Recipient':'alter', 'Commitment Year':'year','Adjusted Amount (Constant USD 2021)':'value'}
OECD_RENAME_DICT = {'Donor':'ego', 'Recipient':'alter', 'TIME_PERIOD':'year','OBS_VALUE':'value'}
INDIA_RENAME_DICT = {'recipientname':'alter', 'usd_disbursment_con':'value'}


RUSSIA_MULTILATERAL_BREAKDOWN = [  # According to https://hub.unido.org/bilateral-development-partners/russian-federation
    'International Development Association [IDA]',
    'World Bank',
    'UNDP', 
    'UNHCR', 
    'UNICEF',
    'WFP',
    'International Bank for Reconstruction and Development [IBRD]',
    'World Health Organisation [WHO]',
    'Food and Agriculture Organisation [FAO]',
    'United Nations Industrial Development Organization [UNIDO]'
]

def load_oda(add_multilateral=False):
    oecd_oda = load_oecd_oda(add_multilateral=add_multilateral)
    if add_multilateral:
        logging.warning("Only supporting multilateral for oecd data")
    china_oda = load_china_oda()
    russia_oda = load_russia_oda()
    india_oda = load_india_oda()
    oda_df = pd.concat([oecd_oda, china_oda, russia_oda, india_oda])
    return oda_df


def load_oecd_oda(add_multilateral=True):
    logging.info("Reading OECD ODA Data (DAC2)")
    oecd_oda = pd.read_csv(ODA_OECD_PATH)
    oecd_oda = oecd_oda[oecd_oda['PRICE_BASE']=='Q']
    if add_multilateral:
        oecd_oda = oecd_oda[(oecd_oda['Measure']=='Imputed multilateral ODA')|(oecd_oda['Measure']=='Official development assistance (ODA), disbursements')]
    else:
        oecd_oda = oecd_oda[oecd_oda['Measure']=='Official development assistance (ODA), disbursements']
    oecd_oda = pd.DataFrame(oecd_oda.groupby(['TIME_PERIOD','Donor','Recipient'])['OBS_VALUE'].sum(numeric_only=True))
    def replace_zero(x):
        if x >= 0:
            return x
        return 0
    oecd_oda['OBS_VALUE'] = oecd_oda['OBS_VALUE'].apply(replace_zero)
    oecd_oda = oecd_oda.reset_index().rename(OECD_RENAME_DICT, axis=1)
    return oecd_oda

def load_china_oda():
    logging.info("Reading China ODA-like Data (Aiddata)")
    china_oda = pd.read_excel(ODA_CHINA_PATH, sheet_name='GCDF_3.0')
    china_oda = china_oda[china_oda['Flow Class']=='ODA-like']
    china_oda = china_oda[~china_oda['Status'].isin(['Cancelled', 'Suspended'])]
    china_oda = pd.DataFrame(china_oda.groupby(['Financier Country', 'Recipient', 'Commitment Year'])['Adjusted Amount (Constant USD 2021)'].sum())
    china_oda['Adjusted Amount (Constant USD 2021)'] = china_oda['Adjusted Amount (Constant USD 2021)'] / 1000000
    china_oda = china_oda.reset_index().rename(AIDDATA_RENAME_DICT, axis=1)

    logging.info("Reading China historic foreign aid data (CIA)")
    china_oda_cia = pd.read_excel(ODA_RUSSIACHINA_PATH, sheet_name='China') # 1085,1986,1988,1989 data
    china_oda_cia = china_oda_cia.reset_index().rename(CIA_RENAME_DICT, axis=1)

    china_oda_total = pd.concat([china_oda, china_oda_cia])
    
    return china_oda_total

def load_russia_oda():  # No multilateral
    logging.info("Reading Russia ODA data")
    russia_oda = pd.read_excel(ODA_RUSSIACHINA_PATH, sheet_name='Russia')
    russia_oda = russia_oda[(russia_oda['Recipient']!='World')&(russia_oda['Type']!='Multilateral Aid')&(russia_oda['Type']!='Bilateral Aid')]
    russia_oda.set_index(['YEAR','DONOR'], inplace=True)
    russia_oda = russia_oda.reset_index().rename(CIA_RENAME_DICT, axis=1)
    return russia_oda

def load_india_oda():
    logging.info("Reading India ODA data from India development finance dataset")
    india_oda = pd.read_excel(ODA_INDIA_PATH)
    india_oda.rename(INDIA_RENAME_DICT, axis=1, inplace=True)
    india_oda['value'] = india_oda['value'] / 1000000
    india_oda['ego'] = 'India'
    india_oda = pd.DataFrame(india_oda.groupby(['ego', 'alter', 'year'])['value'].sum()).reset_index()
    return india_oda

In [233]:
oda_df = load_oda()

In [234]:
oda_df

,year,ego,alter,value,index,Type,Note
0,1960,Austria,All recipients,0.000000,NaN,NaN,NaN
1,1960,Austria,America,2.818579,NaN,NaN,NaN
2,1960,Austria,Argentina,6.263510,NaN,NaN,NaN
3,1960,Austria,Brazil,0.000000,NaN,NaN,NaN
4,1960,Austria,Colombia,0.000000,NaN,NaN,NaN
...,...,...,...,...,...,...,...
458,1993,Russia,Armenia,2.000000,NaN,Total,https://babel.hathitrust.org/cgi/pt?id=mdp.390...
459,1993,Russia,Georgia,2.000000,NaN,Total,https://babel.hathitrust.org/cgi/pt?id=mdp.390...
460,1993,Russia,Serbia,2.000000,NaN,Total,https://babel.hathitrust.org/cgi/pt?id=mdp.390...
461,1993,Russia,Somalia,2.000000,NaN,Total,https://babel.hathitrust.org/cgi/pt?id=mdp.390...


In [212]:
oecd_oda = pd.read_csv(ODA_OECD_PATH)
#oecd_df = load_oecd_oda()

In [214]:
oecd_oda['Donor'].sort_values().unique()

array(['Adaptation Fund', 'African Development Bank',
       'African Development Bank [AfDB]',
       'African Development Fund [AfDF]',
       'Arab Bank for Economic Development in Africa [BADEA]',
       'Arab Fund (AFESD)', 'Arcadia Fund', 'Arcus Foundation',
       'Asian Development Bank', 'Asian Development Bank [AsDB]',
       'Asian Infrastructure Investment Bank [AIIB]', 'Australia',
       'Austria', 'Azerbaijan', 'Belgium', 'Bezos Earth Fund',
       'Bloomberg Family Foundation', 'Bulgaria', 'CGIAR',
       'COVID-19 Response and Recovery Multi-Partner Trust Fund [UN COVID-19 MPTF]',
       'Canada', 'Caribbean Development Bank [CarDB]',
       'Carnegie Corporation of New York',
       'Center of Excellence in Finance [CEF]',
       'Central American Bank for Economic Integration [CABEI]',
       'Central Emergency Response Fund [CERF]',
       'Charity Projects Ltd (Comic Relief)',
       "Children's Investment Fund Foundation", 'Chinese Taipei',
       'Citi Foundation

## China

In [220]:
oda_df[oda_df['ego']=="China (People's Republic of)"]

,year,ego,alter,value,index,Type,Note
0,2000,China (People's Republic of),Afghanistan,1.283228,NaN,NaN,NaN
1,2001,China (People's Republic of),Afghanistan,2.818331,NaN,NaN,NaN
2,2002,China (People's Republic of),Afghanistan,17.983846,NaN,NaN,NaN
3,2003,China (People's Republic of),Afghanistan,17.275664,NaN,NaN,NaN
4,2004,China (People's Republic of),Afghanistan,73.087522,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2351,2017,China (People's Republic of),Zimbabwe,174.435834,NaN,NaN,NaN
2352,2018,China (People's Republic of),Zimbabwe,195.238486,NaN,NaN,NaN
2353,2019,China (People's Republic of),Zimbabwe,93.983185,NaN,NaN,NaN
2354,2020,China (People's Republic of),Zimbabwe,9.468256,NaN,NaN,NaN


In [215]:
load_china_oda().groupby(['ego', 'year']).sum()

/tmp/ipykernel_4027760/909384561.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  load_china_oda().groupby(['ego', 'year']).sum()


value   index
ego                          year                      
China                        1985    610.000000   666.0
                             1986    616.000000  1962.0
                             1988    847.000000  3367.0
                             1989    335.000000  4736.0
                             1993     20.000000   594.0
China (People's Republic of) 2000   1583.901416     0.0
                             2001   3608.105839     0.0
                             2002   2318.250300     0.0
                             2003   4269.757016     0.0
                             2004   2017.161299     0.0
                             2005   3919.641612     0.0
                             2006   5142.095297     0.0
                             2007   6647.433659     0.0
                             2008   6473.583530     0.0
                             2009   6537.767112     0.0
                             2010  18625.746073     0.0
                             2011   6295.085325     0.0
                             2012  10876.802262     0.0
                             2013   8436.085936     0.0
                             2014  11885.876523     0.0
                             2015   9996.483101     0.0
                             2016  14319.607561     0.0
                             2017  14944.126787     0.0
                             2018  13456.433552     0.0
                             2019   7901.926804     0.0
                             2020   3884.607071     0.0
                             2021   4625.274744     0.0

## Russia

In [188]:


russia_oda = load_russia_oda()

In [185]:
china_oda

,ego,alter,year,value
0,China (People's Republic of),Afghanistan,2000,1.283228
1,China (People's Republic of),Afghanistan,2001,2.818331
2,China (People's Republic of),Afghanistan,2002,17.983846
3,China (People's Republic of),Afghanistan,2003,17.275664
4,China (People's Republic of),Afghanistan,2004,73.087522
...,...,...,...,...
2351,China (People's Republic of),Zimbabwe,2017,174.435834
2352,China (People's Republic of),Zimbabwe,2018,195.238486
2353,China (People's Republic of),Zimbabwe,2019,93.983185
2354,China (People's Republic of),Zimbabwe,2020,9.468256


In [176]:
oecd_oda.reset_index()['Donor'].unique()#]

array(['Austria', 'Belgium', 'Canada', 'DAC EU countries',
       'DAC EU countries and EU Institutions', 'DAC Members',
       'DAC countries', 'Denmark', 'EU Institutions', 'France', 'G7',
       'Germany', 'International Development Association [IDA]', 'Italy',
       'Japan', 'Multilaterals organisations', 'Netherlands', 'Norway',
       'Official donors', 'Other Multilaterals', 'Portugal', 'Sweden',
       'Switzerland', 'United Kingdom', 'United Nations', 'United States',
       'World Bank', 'World Bank Group', 'African Development Bank',
       'African Development Fund [AfDF]', 'Australia',
       'Regional Development Banks', 'Non-DAC countries', 'Saudi Arabia',
       'UNDP', 'UNHCR', 'UNICEF', 'UNRWA', 'UNTA', 'WFP', 'Finland',
       'Kuwait', 'New Zealand', 'Other countries', 'United Arab Emirates',
       'Caribbean Development Bank [CarDB]', 'Qatar', 'Arab Fund (AFESD)',
       'Ireland', 'Other multilateral organisations', 'Hungary', 'Poland',
       'Arab Bank for Eco

In [96]:
# No need
#oecd_oda['donor_is_org'] = oecd_oda['DONOR'].apply(lambda x: len(x)>3)
#oecd_oda['recipient_is_org'] = oecd_oda['RECIPIENT'].apply(lambda x: len(x)>3)
#via_mult = oecd_oda[(oecd_oda['donor_is_org'])&(~oecd_oda['recipient_is_org'])]
#via_mult.groupby(['TIME_PERIOD','Donor','Recipient']).sum(numeric_only=True)

## India

,ego,alter,year,value
0,India,Afghanistan,2007,0.000000
1,India,Afghanistan,2008,0.000000
2,India,Afghanistan,2009,0.000000
3,India,Afghanistan,2010,0.000000
4,India,Afghanistan,2011,72.701322
...,...,...,...,...
835,India,Zimbabwe,2011,1.353267
836,India,Zimbabwe,2012,0.191955
837,India,Zimbabwe,2013,1.507391
838,India,Zimbabwe,2014,0.000000


In [26]:


def load_energy():
    
    energy_files = listdir(energy_PATH)
    try:
        energy_files.remove('.ipynb_checkpoints')
    except Exception:
        pass
    print(energy_files)
    dfs=[]
    for file in energy_files:
        if 'syncthing' in file:
            continue
        if '~' in file:
            continue
        logging.info(f'reading {file}')
        energy_data_piece = pd.read_csv(energy_PATH+file, encoding='latin-1', index_col=False)#, index_col=None, header=None, quoting=csv.QUOTE_ALL)
        if 'mirror' in file:  # For mirrored data replace ego an alter labels
            energy_data_piece.rename({'partnerISO':'reporterISO', 'reporterISO':'partnerISO'}, inplace=True, axis=1)
        if energy_data_piece.shape[0] >= 250000:
            logging.error(f'file to big, some data is most probavly cut ({file}:{energy_data_piece.shape[0]})')
        dfs += energy_data_piece,
    energy_df = pd.concat(dfs)
    energy_df['cmdCode']=energy_df['cmdCode'].replace({'TOTAL':np.nan})
    energy_df=energy_df.dropna(subset='cmdCode')
    
    # removing possible duplicates due to mirror operations, keeping largest
    energy_df.sort_values(by='primaryValue', ascending=False, inplace=True)
    duplicates = energy_df[energy_df.duplicated(subset=['refYear', 'reporterISO', 'partnerISO', 'cmdCode'])]
    logging.info(f"Removing {duplicates.shape[1]} rows from {duplicates['reporterISO'].unique()}, years {duplicates['refYear'].unique()}")
    energy_df.drop_duplicates(subset=['refYear', 'reporterISO', 'partnerISO', 'cmdCode'], inplace=True, keep='first')
    
    energy_df=energy_df.groupby(['refYear', 'reporterISO', 'partnerISO']).sum().reset_index()
    return energy_df


In [ ]:

    
def preprocess_energy(df, year_start=1985, year_end=2022, rolling_window=5, normalize=True, test_data=True):

    source_name = preprocess_energy.__name__.split('_')[1]
    EGO_LABEL = 'reporterISO'
    ALTER_LABEL = 'partnerISO'
    YEAR_LABEL = 'refYear'
    VALUE_LABEL = 'primaryValue'
    
    #def preprocessed_energy(df, year_start, rolling_window=5):
    logging.info("Preprocessing energy data")  # basic preprocessing already done
    #removing old or too contemporary data
    df = df[df[YEAR_LABEL].astype(int) >= year_start]
    df = df[df[YEAR_LABEL].astype(int) <= year_end]
    
    df = convert_country_df(df, ALTER_LABEL, standard_to_convert='STATE_en_UN', purge=True)
    df = convert_country_df(df, EGO_LABEL, standard_to_convert='STATE_en_UN', purge=True)

    countries_all = get_all_countries()
    df_triple = df.groupby([YEAR_LABEL, ALTER_LABEL, EGO_LABEL]).sum()
    #df_triple = df_triple[df_triple[VALUE_LABEL]!=0]
    
    df_triple = pd.DataFrame(df_triple[VALUE_LABEL])
    #return df, df_triple
    if test_data:
        test_df(df_triple.reset_index(), source_name, year_start=year_start, year_end=year_end, alter_label=ALTER_LABEL, ego_label=EGO_LABEL, year_label=YEAR_LABEL, value_label=VALUE_LABEL) 
        
    # filling an empty year*country*country dataframe with zeroes
    empty_df=get_empty_country_df(years=df[YEAR_LABEL].unique(), countries_all=countries_all, names=[YEAR_LABEL, EGO_LABEL, ALTER_LABEL])
    
    # merging the df with zero df
    df_triple = empty_df.merge(df_triple,left_index=True, right_index=True, how='outer').fillna(0)
    
    # counting rolling average
    df_triple = df_triple.reset_index().set_index(YEAR_LABEL).groupby([ALTER_LABEL, EGO_LABEL]).rolling(rolling_window, min_periods=1).mean()
    
    df_triple = df_triple.reset_index().set_index([YEAR_LABEL, ALTER_LABEL, EGO_LABEL])
    
    df_triple.index.names = ['year', 'alter', 'ego']
    df_triple = df_triple.rename({VALUE_LABEL:'value'}, axis=1)
    
    # Normalization
    if normalize:
        df_triple['value'] = df_triple['value'] / df_triple['value'].max()
    
    logging.info("Done preprocessing energy Data (COMenergy)")
    return df, df_triple